# ML Model Training & Evaluation

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('seaborn-v0_8-darkgrid')

## Load/Generate Data
Load dataset or generate using CipherClassifier.generate_training_data

In [ ]:
df = pd.read_csv('../data/processed/cipher_dataset.csv')
X = df.drop(columns=['label'])
y = df['label']

## Train-Test Split
80-20 split with stratification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

## Model 1: Random Forest
Train with hyperparameter tuning (GridSearchCV)
Print classification report
Plot confusion matrix heatmap
Plot feature importance bar chart (top 15)

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf_params = {'n_estimators': [50, 100], 'max_depth': [None, 10]}
rf_grid = GridSearchCV(rf, rf_params, cv=3, n_jobs=-1)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print('Random Forest Classification Report:')
print(classification_report(y_test, y_pred_rf))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', 
            xticklabels=best_rf.classes_, yticklabels=best_rf.classes_)
plt.title('Random Forest Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

In [ ]:
rf_importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
rf_importances.head(15).plot(kind='bar')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.ylabel('Importance')
plt.show()

## Model 2: SVM
Train SVM with RBF kernel
Print classification report
Plot confusion matrix

In [ ]:
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print('SVM Classification Report:')
print(classification_report(y_test, y_pred_svm))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Oranges',
            xticklabels=svm.classes_, yticklabels=svm.classes_)
plt.title('SVM Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

## Model 3: Neural Network
Train MLP
Print classification report
Plot confusion matrix

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

print('Neural Network Classification Report:')
print(classification_report(y_test, y_pred_mlp))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_mlp), annot=True, fmt='d', cmap='Greens',
            xticklabels=mlp.classes_, yticklabels=mlp.classes_)
plt.title('Neural Network Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

## Model Comparison
Bar chart comparing accuracy of all 3 models
Cross-validation scores comparison

In [ ]:
acc_rf = accuracy_score(y_test, y_pred_rf)
acc_svm = accuracy_score(y_test, y_pred_svm)
acc_mlp = accuracy_score(y_test, y_pred_mlp)

models = ['Random Forest', 'SVM', 'Neural Network']
accuracies = [acc_rf, acc_svm, acc_mlp]

plt.figure(figsize=(8, 5))
sns.barplot(x=models, y=accuracies, hue=models, legend=False, palette='Set2')
plt.title('Model Accuracy Comparison')
plt.ylim(0, 1)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
plt.show()

## Save Best Model
Save the best performing model using joblib

In [ ]:
best_model = best_rf
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/cipher_classifier.pkl')
print('Best model saved to ../models/cipher_classifier.pkl')

## Summary
Conclusion with best model recommendation